# Face Recognition: MobileFaceNet (Edge AI Architecture)
**Accurate and Efficient Real-Time Face Verification**

## Paper Experiments Pipeline
1. Project Root & GPU Init
2. Main Training (ArcFace + MobileFaceNet)
3. Loss Function Comparison (ArcFace vs CosFace vs SphereFace vs Softmax)
4. Verification Evaluation (EER, TAR@FAR, ROC)
5. Embedding Visualization (t-SNE, Similarity Heatmap)

---
## 1. Project Root & GPU Init

In [1]:
import os
import sys
import tensorflow as tf

# Change directory to project root so relative paths work properly
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../../"))
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)
print(f'Working directory changed to: {os.getcwd()}')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s) Detected: {len(gpus)}')
else:
    print('No GPU detected - running on CPU.')

Working directory changed to: c:\Users\Admin\Desktop\Project_DAT301m
GPU(s) Detected: 1


---
## 2. Main Training: ArcFace + MobileFaceNet


In [2]:
import sys
from models.face_recognition.mobilefacenet.train import main as train_main

sys.argv = [
    'train.py',
    '--dataset_dir', 'ms1m_arcface_dataset/train',
    '--val_dir', 'ms1m_arcface_dataset/val',
    '--test_dir', 'ms1m_arcface_dataset/test',
    '--batch_size', '128',
    '--epochs', '60',
    '--embedding_dim', '512',
    '--loss_type', 'arcface',
    '--lr', '0.001',
    '--weight_decay', '1e-4',
    '--verify_every', '3',
    '--verify_pairs', '3000',
    '--resume', 
]

train_main()

[GPU] Memory Growth enabled. 1 GPU(s) detected.
[Data] Loaded 4748639 train images, 491516 val images across 76504 classes.
[Resume Warning] Checkpoint or metadata files not found. Starting from scratch.
Model: "MobileFaceNet"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 image_input (InputLayer)       [(None, 112, 112, 3  0           []                               
                                )]                                                                
                                                                                                  
 stem_conv (Conv2D)             (None, 56, 56, 64)   1728        ['image_input[0][0]']            
                                                                                                  
 stem_bn (BatchNormalization)   (None, 56, 56, 64)   256         ['stem_conv[0][

: 

---
## 3. Loss Function Comparison


In [ ]:
# Train with CosFace
import importlib
from models.face_recognition.mobilefacenet import train
importlib.reload(train)
from models.face_recognition.mobilefacenet.train import main as train_main

sys.argv = [
    'train.py',
    '--dataset_dir', 'dataset_final/train',
    '--val_dir', 'dataset_final/val',
    '--test_dir', 'dataset_final/test',
    '--batch_size', '256',
    '--epochs', '60',
    '--embedding_dim', '256',
    '--loss_type', 'cosface',
    '--verify_every', '5',
    '--resume', 
]

train_main()

In [ ]:
# Train with SphereFace
import importlib
from models.face_recognition.mobilefacenet import train
importlib.reload(train)
from models.face_recognition.mobilefacenet.train import main as train_main

sys.argv = [
    'train.py',
    '--dataset_dir', 'dataset_final/train',
    '--val_dir', 'dataset_final/val',
    '--test_dir', 'dataset_final/test',
    '--batch_size', '256',
    '--epochs', '60',
    '--embedding_dim', '256',
    '--loss_type', 'sphereface',
    '--verify_every', '5',
    '--resume', 
]

train_main()

In [ ]:
# Train with Softmax (baseline)
import importlib
from models.face_recognition.mobilefacenet import train
importlib.reload(train)
from models.face_recognition.mobilefacenet.train import main as train_main

sys.argv = [
    'train.py',
    '--dataset_dir', 'dataset_final/train',
    '--val_dir', 'dataset_final/val',
    '--test_dir', 'dataset_final/test',
    '--batch_size', '256',
    '--epochs', '60',
    '--embedding_dim', '256',
    '--loss_type', 'softmax',
    '--verify_every', '5',
    '--resume', 
]

train_main()

[GPU] Memory Growth enabled. 1 GPU(s) detected.
[Data] Loaded 36545 train images, 9072 val images across 128 classes.
Model: "MobileFaceNet"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 image_input (InputLayer)       [(None, 112, 112, 3  0           []                               
                                )]                                                                
                                                                                                  
 stem_conv (Conv2D)             (None, 56, 56, 64)   1728        ['image_input[0][0]']            
                                                                                                  
 stem_bn (BatchNormalization)   (None, 56, 56, 64)   256         ['stem_conv[0][0]']              
                                                                   

---
## 4. Verification Evaluation


In [5]:
from shared.eval_verification import main as eval_main

sys.argv = [
    'eval_verification.py',
    '--test_dir', 'dataset_final/test',
    '--weights', 'models/face_recognition/mobilefacenet/results/best_mobilefacenet_backbone.h5',
    '--variant', 'mobilefacenet',  # Placeholder choices or support in eval
    '--img_size', '112',
    '--embedding_dim', '512',
    '--pairs', '10000',
    '--output_dir', 'models/face_recognition/mobilefacenet/results',
]

# Note: We will modify eval_verification.py to add mobilefacenet choice if needed
eval_main()


Face Verification Evaluation
  Variant:    mobilefacenet
  Weights:    models/face_recognition/mobilefacenet/results/best_mobilefacenet_backbone.h5
  Test Dir:   dataset_final/test
  Identities: 108
  Images:     3240
  Pairs:      10000 pos + 10000 neg

[1/3] Extracting embeddings...
[2/3] Building pairs...
[3/3] Computing metrics...

Verification Results (108 unseen identities)
  Identities:     108
  Images used:    3240
  Pairs:          10000 pos + 10000 neg
------------------------------------------------------------
  EER:            0.1349 (13.49%)
  EER Threshold:  0.1604
  AUC:            0.9381
  TAR @ FAR=1e-2: 0.5014 (50.14%)
  TAR @ FAR=1e-3: 0.2209 (22.09%)
  TAR @ FAR=1e-4: 0.1042 (10.42%)
------------------------------------------------------------
  Pos sim (mean): 0.3330 +/- 0.1533
  Neg sim (mean): 0.0333 +/- 0.1164
[Plot] ROC curve saved to: models/face_recognition/mobilefacenet/results\roc_curve.png
[Plot] Score distribution saved to: models/face_recognition/mobi

---
## 5. Embedding Visualization


In [6]:
from shared.visualize_embeddings import main as viz_main

sys.argv = [
    'visualize_embeddings.py',
    '--test_dir', 'dataset_final/test',
    '--weights', 'models/face_recognition/mobilefacenet/results/best_mobilefacenet_backbone.h5',
    '--variant', 'mobilefacenet', # placeholder or support mobilefacenet
    '--img_size', '112',
    '--embedding_dim', '512',
    '--max_ids', '20',
    '--max_images_per_id', '15',
    '--method', 'both',
    '--output_dir', 'models/face_recognition/mobilefacenet/results',
]

viz_main()

Visualizing 20 identities
Total images: 300
Extracting embeddings...
3/3 [==============================] - 1s 71ms/step

Running t-SNE...


TypeError: TSNE.__init__() got an unexpected keyword argument 'n_iter'

In [16]:
import numpy as np
import tensorflow as tf
from PIL import Image
from models.face_recognition.mobilefacenet.mobile_face_net import MobileFaceNet_Backbone

# 1. Cấu hình đường dẫn và tham số
WEIGHTS_PATH = 'models/face_recognition/mobilefacenet/results/best_mobilefacenet_backbone.h5'
IMG_SIZE = 112
EMBEDDING_DIM = 512
THRESHOLD = 0.20  # Ngưỡng tương đồng (thường từ 0.25 - 0.3), cân bằng thì từ 0.2-0.16

# Thay thế bằng đường dẫn tới 2 ảnh của bạn
img_origin_path = r"C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\origin.jpg"  # Ảnh gốc (Origin)
img_sample_path = r"C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\test3.jpg"  # Ảnh kiểm tra (Sample)

# 2. Khởi tạo mô hình và load trọng số đã train
backbone = MobileFaceNet_Backbone(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    embedding_dim=EMBEDDING_DIM,
    normalize_embeddings=True  # L2 normalize embeddings trực tiếp
)
backbone.load_weights(WEIGHTS_PATH)
print("[INFO] Đã load trọng số MobileFaceNet thành công.")

# 3. Hàm tải và tiền xử lý ảnh (giống pipeline huấn luyện)
def preprocess_single_image(path):
    with Image.open(path) as im:
        im = im.convert("RGB")
        # Crop square trung tâm
        w, h = im.size
        side = min(w, h)
        left = (w - side) // 2
        top = (h - side) // 2
        im = im.crop((left, top, left + side, top + side))
        # Resize và chuẩn hóa về khoảng [-1, 1]
        im = im.resize((IMG_SIZE, IMG_SIZE), resample=Image.BILINEAR)
        arr = np.asarray(im, dtype=np.float32)
    arr = (arr - 127.5) / 128.0
    return np.expand_dims(arr, axis=0)  # Thêm chiều batch (1, 112, 112, 3)

# 4. Trích xuất đặc trưng (Embeddings)
img_origin = preprocess_single_image(img_origin_path)
img_sample = preprocess_single_image(img_sample_path)

emb_origin = backbone.predict(img_origin, verbose=0)[0]
emb_sample = backbone.predict(img_sample, verbose=0)[0]

# 5. Tính toán độ tương đồng cosine (Cosine Similarity)
# Do embeddings đã được L2 normalized trong mô hình, Cosine Similarity chính là Dot Product
similarity = np.dot(emb_origin, emb_sample)

# 6. Hiển thị kết quả
print("\n=== KẾT QUẢ ĐỐI CHIẾU ===")
print(f"Ảnh gốc (Origin): {img_origin_path}")
print(f"Ảnh mẫu (Sample): {img_sample_path}")
print(f"Độ tương đồng Cosine: {similarity:.4f}")
print(f"Ngưỡng quyết định (Threshold): {THRESHOLD:.4f}")

if similarity >= THRESHOLD:
    print("👉 KẾT LUẬN: CÙNG MỘT NGƯỜI (MATCH)")
else:
    print("👉 KẾT LUẬN: HAI NGƯỜI KHÁC NHAU (MISMATCH)")


[INFO] Đã load trọng số MobileFaceNet thành công.

=== KẾT QUẢ ĐỐI CHIẾU ===
Ảnh gốc (Origin): C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\origin.jpg
Ảnh mẫu (Sample): C:\Users\Admin\Desktop\Project_DAT301m\test_one_shot\test3.jpg
Độ tương đồng Cosine: 0.4868
Ngưỡng quyết định (Threshold): 0.2000
👉 KẾT LUẬN: CÙNG MỘT NGƯỜI (MATCH)
